# E2 Analysis — OLMo 2 7B Instruct, Pretraining Corpus

**Goal**: Perform a deeper E2 (Compositional Building Block Co-occurrence) analysis for olmo2-7b-instruct on the pretraining corpus (OLMo-Mix-1124). This notebook extends `analyze_e2_olmo2_v1.ipynb` with a focused first step: **deciding which `top_n` value to use for the main analysis**.

**Why `top_n` selection matters**:
- Smaller `top_n` (e.g., 5) = only the most central concepts (topic_core tier); may miss supporting co-occurrences.
- Larger `top_n` (e.g., 20) = includes peripheral concepts, which are often **generic high-frequency tokens** that saturate co-occurrence counts (e.g., 'United States', 'China') and swamp the signal from record-specific concepts.
- The right cut is the one where the E2 metric **discriminates between records** without being dominated by saturation artifacts.

**Scope of this notebook (Part 1: `top_n` selection only)**:
- 6 pilot records: id ∈ {30, 31, 38, 39, 182, 196}
- 4 top_n candidates: **5, 10, 15, 20**
- 3 windows: 100, 500, 1000 tokens
- Decision criteria: growth pattern of E2_cooc / E2_support_score, saturation behavior, and discriminative spread.

**Note on the current data**: These results were produced with `max_clause_freq=None` (API server default = 50,000), which left **~92.3% of queries approximate**. A re-run with `max_clause_freq=500000` is planned. The `top_n` decision here may need re-validation after the re-run, but the growth patterns across `top_n` are expected to be qualitatively stable.

**Reference**: Project Overview 2026-04-09, Section 6.2 (E2 metric definitions).

## Section 0. Setup

Load all four top_n result files and set the shared constants.

Key formula for reference:

$$
\text{E2\_support\_score} = \max_{w \in \{100, 500, 1000\}} \log(1 + \text{E2\_cooc}(w))
$$

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import math
from collections import Counter, defaultdict
from statistics import mean, median

TOP_NS = [5, 10, 15, 20]
WINDOWS = [100, 500, 1000]

BASE_DIR = '../results/olmo2_7b_instruct/e2/gpt-5-mini/pretraining'

data_by_n = {}
for n in TOP_NS:
    with open(f'{BASE_DIR}/e2_cooccurrence_standard_top{n}.json', 'r') as f:
        data_by_n[n] = json.load(f)

n_records = len(data_by_n[TOP_NS[0]])
ids_sorted = sorted(r['id'] for r in data_by_n[TOP_NS[0]])

print(f"Loaded top_n values: {list(data_by_n.keys())}")
print(f"Records per file: {n_records}")
print(f"Record ids: {ids_sorted}")
print(f"Windows tested: {WINDOWS}")
print(f"Total queries = records × pairs × windows:")
for n in TOP_NS:
    pairs = n * (n - 1) // 2
    total_q = n_records * pairs * len(WINDOWS)
    print(f"  top_n={n:>2}: {pairs:>3} pairs/record × {n_records} records × {len(WINDOWS)} windows = {total_q:>5} queries")

Loaded top_n values: [5, 10, 15, 20]
Records per file: 6
Record ids: [30, 31, 38, 39, 182, 196]
Windows tested: [100, 500, 1000]
Total queries = records × pairs × windows:
  top_n= 5:  10 pairs/record × 6 records × 3 windows =   180 queries
  top_n=10:  45 pairs/record × 6 records × 3 windows =   810 queries
  top_n=15: 105 pairs/record × 6 records × 3 windows =  1890 queries
  top_n=20: 190 pairs/record × 6 records × 3 windows =  3420 queries


## Section 1. E2_support_score sensitivity to `top_n`

The most direct question: **for each record, how does the final E2 score (support score) change as we include more concepts?**

Three patterns we care about:
- **Flat / early saturation**: score barely changes from top5 to top20 → the signal is already captured by the top concepts; larger `top_n` adds noise.
- **Monotonic growth**: score keeps growing → new concepts at higher ranks still contribute; we may be cutting too early.
- **Explosive growth**: score jumps by orders of magnitude at higher `top_n` → a single "megapair" of generic concepts dominates; the metric is being hijacked.

Explosive growth is the danger signal: if `E2_cooc` jumps from ~10^5 at top10 to ~10^8 at top20, it means top20 includes some very generic concept pair (e.g., a country name with another country name) whose corpus co-occurrence is astronomical and drowns out record-specific evidence.

In [3]:
print("=" * 90)
print("Section 1. E2_support_score across top_n")
print("=" * 90)
print()
print(f"  {'id':<5}" + "".join(f"{'top'+str(n):>10}" for n in TOP_NS) +
      f"{'Δ(20-5)':>10}  pattern")
print(f"  {'-'*5}" + "".join(f" {'-'*9}" for _ in TOP_NS) + f" {'-'*9}" + "  " + "-" * 35)

for rid in ids_sorted:
    scores = {}
    for n in TOP_NS:
        rec = next((r for r in data_by_n[n] if r['id'] == rid), None)
        scores[n] = rec['e2']['E2_support_score'] if rec else None

    row = f"  {rid:<5}" + "".join(f"{scores[n]:>10.4f}" for n in TOP_NS)
    delta = scores[20] - scores[5]
    row += f"{delta:>+10.4f}  "

    # Classify pattern
    if scores[5] == scores[20]:
        pat = "flat from top5 (early saturate)"
    elif scores[10] == scores[20]:
        pat = "saturate at top10"
    elif scores[15] == scores[20]:
        pat = "saturate at top15"
    elif delta > 3.0:
        pat = "LARGE growth (possible saturation artifact)"
    else:
        pat = "monotonic to top20"
    row += pat
    print(row)

print()
print("  Note: Δ is in log-units. Δ=+3.0 ≈ 20× increase in E2_cooc; Δ=+6.9 ≈ 1000× increase.")

Section 1. E2_support_score across top_n

  id         top5     top10     top15     top20   Δ(20-5)  pattern
  ----- --------- --------- --------- --------- ---------  -----------------------------------
  30      13.0721   13.7290   13.9683   19.4062   +6.3341  LARGE growth (possible saturation artifact)
  31      17.9000   17.9000   17.9000   17.9000   +0.0000  flat from top5 (early saturate)
  38      18.2040   18.2040   18.2040   18.5207   +0.3167  monotonic to top20
  39      16.0338   16.3067   16.3067   16.3067   +0.2729  saturate at top10
  182     15.4293   17.0627   17.0627   17.0627   +1.6334  saturate at top10
  196      6.6657    8.9876    9.1300   12.3103   +5.6446  LARGE growth (possible saturation artifact)

  Note: Δ is in log-units. Δ=+3.0 ≈ 20× increase in E2_cooc; Δ=+6.9 ≈ 1000× increase.


## Section 2. Raw E2_cooc growth (unlogged)

`E2_support_score` uses `log(1 + E2_cooc)` so it **compresses** large count changes. When the underlying `E2_cooc` (max pairwise co-occurrence) jumps from 10^5 to 10^8, the support score only moves by ~7 units in log space, which **understates** the saturation problem.

Here we show the raw `E2_cooc` at w=100 and w=1000 across `top_n` to see the magnitude of the growth directly.

In [4]:
print("=" * 90)
print("Section 2. Raw E2_cooc (max pairwise count) across top_n")
print("=" * 90)

for w in WINDOWS:
    print()
    print(f"  [window = {w}]")
    print(f"  {'id':<5}" + "".join(f"{'top'+str(n):>14}" for n in TOP_NS) + f"  {'ratio(20/5)':>12}")
    print(f"  {'-'*5}" + "".join(f" {'-'*13}" for _ in TOP_NS) + f"  {'-'*12}")

    for rid in ids_sorted:
        counts = {}
        for n in TOP_NS:
            rec = next((r for r in data_by_n[n] if r['id'] == rid), None)
            counts[n] = rec['e2']['metrics_by_window'][str(w)]['E2_cooc'] if rec else 0

        row = f"  {rid:<5}" + "".join(f"{counts[n]:>14,}" for n in TOP_NS)
        ratio = counts[20] / counts[5] if counts[5] > 0 else float('inf')
        ratio_str = f"{ratio:>12,.1f}×" if ratio != float('inf') else f"{'inf':>12}"
        row += f"  {ratio_str}"
        print(row)

print()
print("  Interpretation:")
print("    A ratio > 100× means top20 is including concept pairs whose corpus co-occurrence")
print("    is so large that they dominate the metric. These are typically generic concepts")
print("    like country names, not record-specific evidence.")

Section 2. Raw E2_cooc (max pairwise count) across top_n

  [window = 100]
  id             top5         top10         top15         top20   ratio(20/5)
  ----- ------------- ------------- ------------- -------------  ------------
  30           63,838       146,831       168,352    85,612,759       1,341.1×
  31       12,608,331    12,608,331    12,608,331    12,608,331           1.0×
  38       17,322,560    17,322,560    17,322,560    17,322,560           1.0×
  39        1,248,641     1,496,322     1,496,322     1,496,322           1.2×
  182         189,801       795,402       795,402     4,599,802          24.2×
  196             243         2,420         3,076        37,595         154.7×

  [window = 500]
  id             top5         top10         top15         top20   ratio(20/5)
  ----- ------------- ------------- ------------- -------------  ------------
  30          308,963       602,586       602,586   199,113,351         644.5×
  31       38,078,555    38,078,555    38,

## Section 3. Which concept pair is driving the max? (megapair identification)

`E2_cooc` is a **max** over all pairs, so its value is determined by a **single pair**. If that pair is generic (e.g., `Russia × United States`), the metric is not measuring record-specific evidence — it's measuring how often two common words appear together in the corpus.

For each `top_n`, identify the pair responsible for the max at w=100 and show its components (text, rank, centrality). This reveals whether the metric is dominated by load-bearing concepts or generic filler.

In [5]:
print("=" * 90)
print("Section 3. Megapair (argmax pair) at w=100 per (record, top_n)")
print("=" * 90)

W_FOCUS = 100  # the strictest window, where record-specificity is most visible

for rid in ids_sorted:
    print()
    print(f"  Record id={rid}")
    print(f"  {'top_n':>6}  {'max_count':>12}  {'concept_a (rank, centrality)':<45}  concept_b (rank, centrality)")
    print(f"  {'-'*6}  {'-'*12}  {'-'*44}  {'-'*44}")

    for n in TOP_NS:
        rec = next((r for r in data_by_n[n] if r['id'] == rid), None)
        if rec is None:
            continue
        pairs = rec['e2']['pairwise_cooccurrence']['pairs']
        # find argmax at w=100
        best = max(pairs, key=lambda p: p['counts_by_window'][str(W_FOCUS)]['count'])
        best_count = best['counts_by_window'][str(W_FOCUS)]['count']

        ca = best['concept_a']
        cb = best['concept_b']
        a_label = f"{ca['text'][:28]!r} (r={ca['rank']}, {ca['centrality'][:12]})"
        b_label = f"{cb['text'][:28]!r} (r={cb['rank']}, {cb['centrality'][:12]})"

        print(f"  {n:>6}  {best_count:>12,}  {a_label:<45}  {b_label}")

print()
print("  Red flag: if the megapair at top20 is suddenly a pair of generic concepts")
print("  (both rank > 10 or centrality = 'peripheral'/'supporting'), that's saturation.")

Section 3. Megapair (argmax pair) at w=100 per (record, top_n)

  Record id=30
   top_n     max_count  concept_a (rank, centrality)                   concept_b (rank, centrality)
  ------  ------------  --------------------------------------------  --------------------------------------------
       5        63,838  'political reform' (r=4, primary)              'human rights' (r=5, primary)
      10       146,831  'human rights' (r=5, primary)                  'Assad regime' (r=7, primary)
      15       168,352  'Assad regime' (r=7, primary)                  'Islamic State' (r=13, primary)
      20    85,612,759  'Iran' (r=19, supporting)                      'United States' (r=20, supporting)

  Record id=31
   top_n     max_count  concept_a (rank, centrality)                   concept_b (rank, centrality)
  ------  ------------  --------------------------------------------  --------------------------------------------
       5    12,608,331  'Crimea' (r=1, topic_core)              

## Section 4. Discriminative spread across records

For a metric to be useful for downstream analysis (e.g., Type B vs Type C classification), it must **differentiate records from each other**. A `top_n` where all records look similar is useless.

We measure this two ways:
- **Spread of E2_support_score**: range (max - min) across the 6 records. Larger = better discrimination.
- **Coefficient of variation (CV) of raw E2_cooc**: std/mean across records. Larger = more record-to-record variation.

We want the `top_n` where records are **differentiable**, not where they collapse to similar saturated values.

In [6]:
print("=" * 90)
print("Section 4. Discriminative spread across records, per top_n")
print("=" * 90)
print()

def _stdev(xs):
    if len(xs) < 2:
        return 0.0
    m = sum(xs) / len(xs)
    return (sum((x - m) ** 2 for x in xs) / (len(xs) - 1)) ** 0.5

print(f"  {'top_n':>6}  {'E2_support_range':>17}  {'E2_support_std':>15}  "
      f"{'E2_cooc_mean(w100)':>19}  {'E2_cooc_CV(w100)':>17}")
print(f"  {'-'*6}  {'-'*17}  {'-'*15}  {'-'*19}  {'-'*17}")

for n in TOP_NS:
    supports = [r['e2']['E2_support_score'] for r in data_by_n[n]]
    coocs_w100 = [r['e2']['metrics_by_window']['100']['E2_cooc'] for r in data_by_n[n]]

    supp_range = max(supports) - min(supports)
    supp_std = _stdev(supports)
    cooc_mean = sum(coocs_w100) / len(coocs_w100)
    cooc_std = _stdev(coocs_w100)
    cv = cooc_std / cooc_mean if cooc_mean > 0 else 0.0

    print(f"  {n:>6}  {supp_range:>17.4f}  {supp_std:>15.4f}  "
          f"{cooc_mean:>19,.0f}  {cv:>17.2f}")

print()
print("  Notes:")
print("    - E2_support_range/std:  spread of the log-scaled support score across 6 records.")
print("    - E2_cooc_CV:            raw coefficient of variation; high CV = records differ a lot.")
print("    - We prefer the top_n that preserves separation (high spread) without pathological CV.")

Section 4. Discriminative spread across records, per top_n

   top_n   E2_support_range   E2_support_std   E2_cooc_mean(w100)   E2_cooc_CV(w100)
  ------  -----------------  ---------------  -------------------  -----------------
       5            11.5383           4.2871            5,238,902               1.47
      10             9.2164           3.5101            5,395,311               1.40
      15             9.0740           3.4368            5,399,007               1.40
      20             7.0959           2.5040           20,279,562               1.61

  Notes:
    - E2_support_range/std:  spread of the log-scaled support score across 6 records.
    - E2_cooc_CV:            raw coefficient of variation; high CV = records differ a lot.
    - We prefer the top_n that preserves separation (high spread) without pathological CV.


## Section 5. Nonzero-fraction stability

Besides the max count, another signal is `E2_nonzero_frac` — the fraction of concept pairs with any co-occurrence in the corpus. Unlike the max, this is **less sensitive to saturation** because it only cares about "any vs none" at the pair level.

If `E2_nonzero_frac` is stable across `top_n` for each record, that's evidence the added concepts are "more of the same" (similar co-occurrence density) rather than saturating the metric.

In [7]:
print("=" * 90)
print("Section 5. E2_nonzero_frac at w=100 across top_n")
print("=" * 90)
print()
print(f"  {'id':<5}" + "".join(f"{'top'+str(n):>10}" for n in TOP_NS) +
      f"{'Δ(20-5)':>10}")
print(f"  {'-'*5}" + "".join(f" {'-'*9}" for _ in TOP_NS) + f" {'-'*9}")

for rid in ids_sorted:
    fracs = {}
    for n in TOP_NS:
        rec = next((r for r in data_by_n[n] if r['id'] == rid), None)
        fracs[n] = rec['e2']['metrics_by_window']['100']['E2_nonzero_frac'] if rec else None
    row = f"  {rid:<5}" + "".join(f"{fracs[n]:>10.4f}" for n in TOP_NS)
    delta = fracs[20] - fracs[5]
    row += f"{delta:>+10.4f}"
    print(row)

print()
print("  Interpretation: a stable nonzero_frac across top_n means the corpus-coverage")
print("  signal is robust. Big swings would suggest top_n is changing the qualitative")
print("  nature of the evidence (not just its magnitude).")

Section 5. E2_nonzero_frac at w=100 across top_n

  id         top5     top10     top15     top20   Δ(20-5)
  ----- --------- --------- --------- --------- ---------
  30       0.4000    0.6889    0.6952    0.7684   +0.3684
  31       1.0000    0.8889    0.9333    0.9368   -0.0632
  38       0.6000    0.6222    0.6857    0.7228   +0.1228
  39       1.0000    0.8889    0.7143    0.6349   -0.3651
  182      0.7000    0.5778    0.4762    0.5464   -0.1536
  196      0.3000    0.3556    0.3048    0.3763   +0.0763

  Interpretation: a stable nonzero_frac across top_n means the corpus-coverage
  signal is robust. Big swings would suggest top_n is changing the qualitative
  nature of the evidence (not just its magnitude).


## Section 6. Per-concept centrality tier distribution by `top_n`

Stage 2 (`e2_rank_concepts.py`) assigns each concept a centrality tier: `topic_core`, `primary`, `supporting`, or `peripheral`. As we increase `top_n`, we progressively include lower-tier concepts.

This section shows which tiers are included at each `top_n` cut, so we can see exactly what we're gaining/losing. The intuition: the ideal `top_n` likely sits at the boundary between **load-bearing** (topic_core + primary) and **peripheral** concepts.

In [8]:
print("=" * 90)
print("Section 6. Centrality tier distribution across top_n (all records pooled)")
print("=" * 90)
print()

TIER_ORDER = ['topic_core', 'primary', 'supporting', 'peripheral', '']

print(f"  {'top_n':>6}  " + "  ".join(f"{t:>11}" for t in TIER_ORDER) + "  total")
print(f"  {'-'*6}  " + "  ".join("-" * 11 for _ in TIER_ORDER) + "  -----")

for n in TOP_NS:
    tier_counts = Counter()
    total = 0
    for r in data_by_n[n]:
        for c in r['e2'].get('ranked_concepts', []):
            tier_counts[c.get('centrality', '')] += 1
            total += 1
    row = f"  {n:>6}  " + "  ".join(f"{tier_counts.get(t, 0):>11}" for t in TIER_ORDER) + f"  {total:>5}"
    print(row)

print()
print("  (Counts are aggregated over all 6 records.)")
print()
print("  Decision heuristic: if going from top_n=X to top_n=X+5 adds mostly 'peripheral'")
print("  concepts AND triggers huge E2_cooc jumps (Section 2), X is the right cut.")

Section 6. Centrality tier distribution across top_n (all records pooled)

   top_n   topic_core      primary   supporting   peripheral               total
  ------  -----------  -----------  -----------  -----------  -----------  -----
       5           17           13            0            0            0     30
      10           17           43            0            0            0     60
      15           17           69            4            0            0     90
      20           17           81           22            0            0    120

  (Counts are aggregated over all 6 records.)

  Decision heuristic: if going from top_n=X to top_n=X+5 adds mostly 'peripheral'
  concepts AND triggers huge E2_cooc jumps (Section 2), X is the right cut.


## Section 7. Decision summary

Pulling Sections 1–6 together to see which `top_n` best balances:
1. **Capturing load-bearing concepts** (we don't want to cut too early and miss them).
2. **Avoiding saturation artifacts** (we don't want generic concept pairs dominating the max).
3. **Preserving record-to-record discrimination** (the metric has to differentiate records).

This cell prints the key numbers side-by-side for each `top_n` candidate and flags each with a qualitative verdict based on simple thresholds. **These thresholds are heuristic and should be sanity-checked against the tables above.**

In [9]:
print("=" * 90)
print("Section 7. top_n decision summary")
print("=" * 90)
print()

summary_rows = []
for n in TOP_NS:
    supports = [r['e2']['E2_support_score'] for r in data_by_n[n]]
    coocs_w100 = [r['e2']['metrics_by_window']['100']['E2_cooc'] for r in data_by_n[n]]
    fracs_w100 = [r['e2']['metrics_by_window']['100']['E2_nonzero_frac'] for r in data_by_n[n]]

    # max_count_order_of_magnitude
    max_cooc = max(coocs_w100) if coocs_w100 else 0
    log_max = math.log10(max_cooc) if max_cooc > 0 else 0

    summary_rows.append({
        'top_n': n,
        'supp_range': max(supports) - min(supports),
        'supp_median': median(supports),
        'max_cooc': max_cooc,
        'log10_max_cooc': log_max,
        'nz_frac_median': median(fracs_w100),
    })

print(f"  {'top_n':>6}  {'supp_range':>11}  {'supp_median':>12}  "
      f"{'max_E2_cooc':>14}  {'log10(max)':>11}  {'nz_frac_med':>12}  verdict")
print(f"  {'-'*6}  {'-'*11}  {'-'*12}  {'-'*14}  {'-'*11}  {'-'*12}  {'-'*40}")

for r in summary_rows:
    # heuristic verdict
    if r['log10_max_cooc'] >= 8:
        verdict = "SATURATED (max_E2_cooc ≥ 10^8)"
    elif r['supp_range'] < 1.0:
        verdict = "LOW DISCRIMINATION (range < 1.0)"
    else:
        verdict = "candidate"
    print(f"  {r['top_n']:>6}  {r['supp_range']:>11.4f}  {r['supp_median']:>12.4f}  "
          f"{r['max_cooc']:>14,}  {r['log10_max_cooc']:>11.2f}  {r['nz_frac_median']:>12.4f}  {verdict}")

print()
print("  Reading the verdict column:")
print("    SATURATED         → max pair count ≥ 10^8; a generic megapair likely dominates.")
print("    LOW DISCRIMINATION → records look too similar to tell apart.")
print("    candidate         → plausible cut; pick by combining with Section 3 megapair inspection.")
print()
print("  Next step: use Section 3 to inspect the megapair for each 'candidate' top_n and confirm")
print("  it corresponds to load-bearing (topic_core/primary) concepts, not peripheral generics.")

Section 7. top_n decision summary

   top_n   supp_range   supp_median     max_E2_cooc   log10(max)   nz_frac_med  verdict
  ------  -----------  ------------  --------------  -----------  ------------  ----------------------------------------
       5      11.5383       15.7315      17,322,560         7.24        0.6500  candidate
      10       9.2164       16.6847      17,322,560         7.24        0.6555  candidate
      15       9.0740       16.6847      17,322,560         7.24        0.6905  candidate
      20       7.0959       17.4813      85,612,759         7.93        0.6788  candidate

  Reading the verdict column:
    SATURATED         → max pair count ≥ 10^8; a generic megapair likely dominates.
    LOW DISCRIMINATION → records look too similar to tell apart.
    candidate         → plausible cut; pick by combining with Section 3 megapair inspection.

  Next step: use Section 3 to inspect the megapair for each 'candidate' top_n and confirm
  it corresponds to load-bearing

## Section 8. Tentative decision (to be confirmed)

**Decision to record after reviewing the tables above**:

- [ ] Chosen `top_n` for main analysis: **TBD**
- [ ] Rationale (reference Sections 1–7):
- [ ] Sensitivity `top_n` values to report alongside (if any):

**Caveat**: All `top_n` analyses in this notebook use the `max_clause_freq=None` (server default 50,000) run, which produced ~92.3% approximate counts. After the re-run with `max_clause_freq=500000`, re-execute this notebook and verify that the chosen `top_n` remains appropriate.

**Next notebooks in this series**:
- `analyze_e2_olmo2_7b_instruct_v2.ipynb` — per-record E2 profile at the chosen `top_n`
- Further versions for E1–E2 joint analysis, concept-level centrality × co-occurrence, and archetype case studies (matching the structure of `analyze_e2_olmo2_v1.ipynb`).